# 👔 Supervisor Pattern

## What is the Supervisor Pattern?
A **supervisor** agent:
1. Receives the task
2. Decides which specialist to delegate to
3. Reviews results
4. Either delegates again or finishes

## Mental Model: Manager + Team
```
Manager (Supervisor)
├── Programmer (CodeAgent)
├── Researcher (SearchAgent)
└── Designer (UIAgent)
```

Manager understands the big picture.
Specialists are deep experts.


In [ ]:
# ── Supervisor Pattern with LangGraph ─────────────────────────────────
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage
from pydantic import BaseModel, Field

class SupervisorDecision(BaseModel):
    next_agent: Literal['coder', 'researcher', 'FINISH'] = Field(
        description='Which agent should work next, or FINISH if done'
    )
    reasoning: str = Field(description='Why this choice')

class TeamState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    task: str
    results: list[str]
    next_agent: str

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
supervisor_llm = llm.with_structured_output(SupervisorDecision)

def supervisor(state: TeamState) -> dict:
    results_str = '\n'.join(state['results']) if state['results'] else 'None yet'
    decision = supervisor_llm.invoke([
        SystemMessage(content='''You are a supervisor managing: coder, researcher.
Delegate tasks and say FINISH when the task is complete.'''),
        HumanMessage(content=f'Task: {state["task"]}\nResults so far: {results_str}\nWhat next?')
    ])
    return {'next_agent': decision.next_agent}

def coder(state: TeamState) -> dict:
    response = llm.invoke([HumanMessage(content=f'Write code for: {state["task"]}. Be brief.')])
    return {'results': [f'[Code]: {response.content}']}

def researcher(state: TeamState) -> dict:
    response = llm.invoke([HumanMessage(content=f'Research: {state["task"]}. Give 2 facts.')])
    return {'results': [f'[Research]: {response.content}']}

def route_next(state: TeamState) -> str:
    return state['next_agent']  # 'coder', 'researcher', or 'FINISH'

graph = StateGraph(TeamState)
graph.add_node('supervisor', supervisor)
graph.add_node('coder', coder)
graph.add_node('researcher', researcher)
graph.set_entry_point('supervisor')
graph.add_conditional_edges('supervisor', route_next, {
    'coder': 'coder',
    'researcher': 'researcher',
    'FINISH': END
})
graph.add_edge('coder', 'supervisor')      # back to supervisor
graph.add_edge('researcher', 'supervisor')  # back to supervisor
app = graph.compile()

result = app.invoke({
    'messages': [],
    'task': 'Build a Python function to sort a list',
    'results': [],
    'next_agent': ''
})
print('All results:')
for r in result['results']:
    print(f'  {r[:100]}')

## 🎯 Interview Questions

1. **What is the supervisor pattern?**
2. **How does the supervisor decide which agent to call?**
3. **How do you prevent infinite loops in supervisor pattern?**
4. **When would you use supervisor vs sequential pipeline?**

> Answer these before moving to the next module.

## 💪 Mini Exercise

**Exercise**: Based on what you learned in this module:

1. Re-run all code cells with your own inputs
2. Modify one code cell to add new functionality
3. Answer the interview questions above from memory

**Mini Project**: Build a small application that combines the concepts from this module.


## 📚 Summary

In this module you learned:
- The core concepts of **Supervisor Pattern — Hierarchical Agent Control**
- How to implement them with modern LangChain/LangGraph APIs
- Common mistakes and how to avoid them
- Interview-level understanding of why each component exists

**Next**: Continue to the next module to build on these foundations.

---
*Part of the LangChain & LangGraph Master Course*
